# Full Phase-0: CIFAR-10 / ResNet-18 trajectory signatures
This notebook runs 30 training jobs on a GPU and saves completed runs plus epoch checkpoints in Google Drive. The first cell downloads the latest bundled source. The training cell reports startup, batch, epoch, and heartbeat progress; `progress.json` records its current stage. If Colab disconnects, run all cells again to resume from the latest saved epoch. Colab can end runtimes independently of notebook code. This notebook does **not** issue a scientific verdict.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
from pathlib import Path
import hashlib, os, shutil, subprocess, sys, time, urllib.request, zipfile
source_zip = Path('/content/phase0_source.zip')
source_url = 'https://raw.githubusercontent.com/anjaya02/TrajSig/main/phase0_source.zip?refresh=' + str(time.time_ns())
print('Downloading current source from', source_url, flush=True)
try:
    urllib.request.urlretrieve(source_url, source_zip)
except Exception as exc:
    print('GitHub download failed; upload phase0_source.zip manually:', exc)
    uploaded = files.upload()
    if 'phase0_source.zip' not in uploaded:
        raise RuntimeError('Expected phase0_source.zip')
project = Path('/content/trajectory-signature-phase0')
if project.exists():
    shutil.rmtree(project)
project.mkdir()
print('Source SHA256:', hashlib.sha256(source_zip.read_bytes()).hexdigest(), flush=True)
with zipfile.ZipFile(source_zip) as archive:
    if b'CHECKING saved runs' not in archive.read('src/trajsig/train.py'):
        raise RuntimeError('Downloaded an old source ZIP. Refresh the GitHub source and rerun this cell.')
    archive.extractall(project)
os.chdir(project)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[test]'], check=True)
print('Installed project from', project, flush=True)

In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable. Select Runtime > Change runtime type > T4 GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

In [ ]:
output = Path('/content/drive/MyDrive/trajsig_phase0_results')
output.mkdir(parents=True, exist_ok=True)
print('Persistent output:', output, flush=True)
progress_file = output / 'progress.json'
if progress_file.exists():
    print('Saved progress from prior session:', progress_file.read_text(), flush=True)
print('Watch this cell for BATCH and PROGRESS lines. Drive progress file:', progress_file, flush=True)
process = subprocess.Popen([sys.executable, '-u', '-m', 'trajsig', 'train', '--config', 'configs/full.yaml', '--output', str(output)])
while True:
    try:
        code = process.wait(timeout=60)
        if code:
            raise RuntimeError(f'Training exited with code {code}; inspect the output above')
        break
    except subprocess.TimeoutExpired:
        print('HEARTBEAT: training process is alive at', time.strftime('%H:%M:%S'), flush=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'trajsig', 'audit', '--config', 'configs/full.yaml', '--output', str(output)], check=True)
archive = output / 'phase0_training_artifacts.zip'
subprocess.run([sys.executable, '-m', 'trajsig', 'package', '--config', 'configs/full.yaml', '--output', str(output), '--destination', str(archive)], check=True)
print('FULL TRAINING AND AUDIT COMPLETE')
print('Return this file for Stage 5 analysis:', archive)
print('Size (GiB):', archive.stat().st_size / 1024**3)